<!-- R01 -->
# Ripristino dei risultati di riferimento
Computer Vision · Lezione 14. Percorso separato dal fit: carica i checkpoint già addestrati e verifica la validation pulita. Non produce una nuova misura di training. Eseguire R01–R08 in ordine dalla cartella `recupero/`. I tempi LP/FT provengono dal computer di riferimento, eventuali nuove misure di inferenza dal computer corrente.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

In [ ]:
# R02
from pathlib import Path
import sys,json,shutil,copy
# Avvio Colab nello stesso notebook: non aggiunge né rinumera celle Cxx.
import os, importlib.util, importlib.metadata, subprocess, zipfile, io
IN_COLAB=importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import files
    _root=Path("/content/ComputerVision_Lezione14")
    _package=_root/"Transfer_Learning_Studio_studenti"
    if not (_package/"supporto/transfer_utils.py").exists():
        print("Carica Transfer_Learning_Studio_CV2026_studenti.zip, senza estrarlo sul browser.")
        _uploaded=files.upload()
        _archives=[v for k,v in _uploaded.items() if k.endswith(".zip")]
        if len(_archives)!=1: raise RuntimeError("Caricare un solo ZIP della Lezione 14.")
        _root.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(_archives[0])) as _z:
            for _entry in _z.infolist():
                if not (_root/_entry.filename).resolve().is_relative_to(_root.resolve()):
                    raise RuntimeError("Percorso non valido nello ZIP.")
            _z.extractall(_root)
        if not (_package/"supporto/transfer_utils.py").exists():
            raise RuntimeError("Lo ZIP caricato non è il pacchetto studenti della Lezione 14.")
    # Conserva la coppia torch/torchvision del runtime, non installa requirements.txt.
    import torch, torchvision
    _constraint=_root/"vincoli_colab.txt"
    _constraint.write_text(f"torch=={torch.__version__}\ntorchvision=={torchvision.__version__}\n")
    if importlib.util.find_spec("timm") is None:
        subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--constraint",str(_constraint),"timm==1.0.30"])
    os.chdir(_package/"recupero")
    print("Colab: protocollo CPU; torch/torchvision conservati; directory:",Path.cwd())
import torch
BASE=Path.cwd().parent
sys.path.insert(0,str(BASE/'supporto'))
import transfer_utils as u
from transfer_utils import head_of
ENV=u.setup();REC=BASE/'recupero'
OUT=Path.cwd()/'output_recupero';OUT.mkdir(exist_ok=True)
print('Ambiente corrente:',ENV)

In [ ]:
# R03
record=json.loads((REC/'metriche_finali.json').read_text())
manifest=json.loads((BASE/'manifest.json').read_text())
REFERENCE_ENV=record['environment'];RESULTS_ORIGIN='Pesi e training importati dalla corsa di riferimento; nuova inferenza sul computer corrente'
data=u.load_data(BASE);assert data['sha256']==manifest['data_sha256']
for k,v in manifest['weights'].items():assert u.sha(BASE/'pesi'/v['file'])==v['sha256']
for file,sha in manifest['reference_files'].items():assert u.sha(REC/file)==sha
results=record['models'];print(RESULTS_ORIGIN);print('Ambiente del training:',REFERENCE_ENV)

In [ ]:
# R04
models={}
for name in ['resnet18','deit_tiny']:
    m,_=u.make_model(name,BASE)
    delta=torch.load(REC/f'ft_delta_{name}.pt',weights_only=True,map_location='cpu')
    state=m.state_dict();state.update(delta);m.load_state_dict(state,strict=True);m.eval()
    models[name]=m
    shutil.copy2(REC/f'ft_delta_{name}.pt',OUT/f'ft_delta_{name}.pt')
    print(name,'delta caricato, checkpoint FT',results[name]['ft']['selected_epoch'])

In [ ]:
# R05
for name,m in models.items():
    val,seconds=u.eval_model(m,data['images'][data['val_idx']],data['target'][data['val_idx']])
    ref=results[name]['ft']['validation']
    assert val['predictions']==ref['predictions']
    assert abs(val['ce']-ref['ce'])<1e-5
    print(name,'verifica',val['correct'],'/360; tempo inferenza locale [s]',seconds)

In [ ]:
# R06
for name,r in results.items():
    print(name,'LP',r['lp']['validation']['accuracy'],'FT',r['ft']['validation']['accuracy'])
    print('Tempi di riferimento: LP',r['lp']['total_seconds'],'s; FT',r['ft']['total_seconds'],'s')
    for kind,v in r['stress'].items():print(kind,v['correct'],'/360',v.get('delta_accuracy_pp',0),'pp')

<!-- R07 -->
## Ritorno alle attività
Le misure sono importate e devono essere etichettate così. Per rieseguire stress/challenge, si possono copiare le celle C29–C35 del notebook studenti nello stesso kernel, senza ricreare i modelli con C09. **C25 richiede una nuova misura locale di latenza:** non confrontarla con un tempo di training importato senza separarne la provenienza. Per completare la scheda usa il JSON/CSV di riferimento e annota origine e hardware. Questo recupero non riprende lo stato Adam né dimostra una nuova esecuzione del fit.

In [ ]:
# R08
summary={'results_origin':RESULTS_ORIGIN,'reference_environment':REFERENCE_ENV,'current_environment':ENV,'reference_data_sha256':record['data_sha256'],'reference_metrics_file':str(REC/'metriche_finali.json'),'clean_predictions_verified':True,'test_evaluated':False}
(OUT/'verifica_recupero.json').write_text(json.dumps(summary,indent=2))
print('Verifica e provenienza:',OUT/'verifica_recupero.json')